In [ ]:
import re
from collections import Counter

import json
from pathlib import Path

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from readers.rais_reader import RaisReader

from discovery.ftp_client import FTPClient
from discovery.filename_parser import RaisFilenameParser
from discovery.dataset_selector import RaisDatasetSelector

from profiling.schema_profiler import RaisSchemaProfiler

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

## Configuração de Paths

In [ ]:
# Data paths
client_path = Path("/pdet/microdados/RAIS")
extracted_data_path = Path("data")
dict_dir = Path("dictionaries")

def save_json(path, doc):
    with open(path, "w") as file:
        json.dump(doc, file)

def load_json(path):
    def hint_tuples(item):
        if isinstance(item, dict):
            return {int(k) if k.isdigit() else k: v for k, v in item.items()}
        return item

    with open(path, "r") as file:
        arquivos = json.load(file, object_hook=hint_tuples)
    return arquivos

## Pré-processamento

### Parsing dos arquivos e seleção dos relevantes

In [ ]:
client = FTPClient("ftp.mtps.gov.br")
client.connect()

arquivos = {}

for year in range(2020, 2021):
    try:
        arquivos[year] = client.list_directory(str(client_path.joinpath(str(year))))

        parser = RaisFilenameParser()
        parsed = [parser.parse(a) for a in arquivos[year]]

        selector = RaisDatasetSelector()
        arquivos[year] = selector.select_valid_datasets(parsed)

    except Exception as e:
        print(e)
        continue

client.disconnect()

### Extração do schema dos dados

In [ ]:
profiler = RaisSchemaProfiler()
profiles = []

for year in range(2020, 2021):
    path = extracted_data_path.joinpath(str(year))

    if not path.exists():
        continue

    arquivos_extracted = [item for item in path.iterdir() if item.is_file()]

    for a in arquivos_extracted:
        profile = profiler.profile_schema(a)

        arquivo = next((item for item in arquivos[year] if item['stem'] == profile['file_name'].split('.')[0]))
        arquivo['profile'] = profile

save_json("data/arquivos.json", arquivos)

## Leitura e Processamento

### Leitura

In [ ]:
arquivos = load_json("data/arquivos.json")
reader = RaisReader()
data = {}
ds_type = "vinculo"

for year in range(2020, 2021):
    n_arqs = 0
    data[year] = {}

    for arquivo in arquivos[year]:
        n_arqs += 1

        if not arquivo["dataset_type"] == ds_type:
            continue

        if arquivo.get("profile"):
            profile = arquivo["profile"]
            data[year][arquivo["stem"]] = {}

            reader.configure(
                encoding=profile["encoding"],
                separator=profile["separator"]
            )

            filepath = (extracted_data_path / str(year) / profile["file_name"])
            print(filepath)

            for i, chunk in enumerate(reader.read(filepath)):
                data[year][arquivo['stem']][i] = chunk

                if i == 1:
                    break

### Normalização

In [ ]:
from normalization.normalizer import RaisNormalizer

normalizer = RaisNormalizer()
dfs = []

for regiao in data[2020]:
    for i in range(len(data[2020][regiao])):
        chunk = data[2020][regiao][i]
        chunk = normalizer.normalize(chunk, dataset_type="vinculo")
        dfs.append(chunk)

df = pd.concat(dfs).reset_index()
df = df.drop("sigla_uf", axis=1)
df = df.drop("categoria_trabalhador", axis=1)

### Seleção de colunas

In [ ]:
municipios = pd.read_csv("dictionaries/id_municipio.csv")
municipios = municipios[["id_municipio_6", "sigla_uf", "nome_uf"]].drop_duplicates()
municipios = municipios.rename(columns={"id_municipio_6": "id_municipio"})

df = df.merge(municipios, on="id_municipio", how="left")

cols = [
    "sexo",
    "raca_cor",
    "faixa_etaria",
    "grau_instrucao_apos_2005",
    "sigla_uf",
    "tamanho_estabelecimento",
    "tipo_vinculo",
    # "categoria_trabalhador",
    "subsetor_ibge",
    "valor_remuneracao_media"
]

df = df[cols]
df

### Tradução dos dicionários

In [ ]:
df["valor_remuneracao_media"] = pd.to_numeric(df["valor_remuneracao_media"].str.replace(',', '.'))
df = df[df["valor_remuneracao_media"] > 0].copy()

categorical_cols = [
    "sexo",
    "raca_cor",
    "faixa_etaria",
    "grau_instrucao_apos_2005",
    "tamanho_estabelecimento",
    "tipo_vinculo",
    # "categoria_trabalhador",
    "subsetor_ibge"
]

def apply_dictionary(df, column, dictionaries_path):
    path = dictionaries_path / f"{column}.csv"

    dictionary = pd.read_csv(path)
    dictionary = dictionary.copy()
    dictionary["chave"] = pd.to_numeric(dictionary["chave"], errors="coerce")
    dictionary = dictionary.dropna(subset=["chave"])
    dictionary["chave"] = dictionary["chave"].astype(int)

    mapping = dict(zip(dictionary["chave"], dictionary["valor"]))
    df[column] = df[column].map(mapping)

    return df

for col in categorical_cols:
    df = apply_dictionary(df, col, dict_dir)

df

### Limpeza

In [ ]:
for col in categorical_cols:
    print("\n")
    print(df[col].value_counts().head(10))

INVALID_VALUES = [
    "Não identificado",
    "IGNORADO",
    "Código não encontrado nos dicionários oficiais."
]

for col in [
    "sexo",
    "raca_cor",
    "faixa_etaria",
    "grau_instrucao_apos_2005",
    "tamanho_estabelecimento",
    "tipo_vinculo",
    # "categoria_trabalhador",
    "subsetor_ibge",
]:
    df = df[~df[col].isin(INVALID_VALUES)]

df = df.dropna()
df

df.to_csv("data/rais_2020.csv")

## Subset Mining

### Utils

In [ ]:
!pip install pysubgroup

In [ ]:
import pysubgroup as ps

def create_target(df, percentile):
    threshold = df["valor_remuneracao_media"].quantile(percentile)

    target_col = f"target_p{int(percentile * 100)}"

    df[target_col] = (df["valor_remuneracao_media"] >= threshold)

    print(f"\nPercentil {percentile:.0%}")
    print(f"Threshold: R$ {threshold:.2f}")
    print(df[target_col].value_counts())
    print(df[target_col].value_counts(normalize=True))

    return target_col, threshold

### Carregamento dos dados

In [ ]:
df_2024 = pd.read_csv("data/rais_2024.csv")
df_2023 = pd.read_csv("data/rais_2023.csv")
df_2022 = pd.read_csv("data/rais_2022.csv")
df_2021 = pd.read_csv("data/rais_2021.csv")
df_2020 = pd.read_csv("data/rais_2020.csv")

dfs = {
    2020: df_2020,
    2021: df_2021,
    2022: df_2022,
    2023: df_2023,
    2024: df_2024
}

## Experimento 1 - Evolução dos Top Subgrupos (2020-2024)

In [ ]:
def prepare_mining_df(df, percentile=0.99):
    df = df.copy()

    threshold = df["valor_remuneracao_media"].quantile(percentile)
    df["target"] = (df["valor_remuneracao_media"] >= threshold)

    mining_df = df.copy()

    for col in [
        "sexo",
        "sigla_uf",
        "raca_cor",
        "faixa_etaria",
        "tipo_vinculo",
        "subsetor_ibge",
        "tamanho_estabelecimento",
        "grau_instrucao_apos_2005",
    ]:
        mining_df[col] = mining_df[col].astype(str)

    search_space = ps.create_selectors(mining_df, ignore=["valor_remuneracao_media", "target"])

    return mining_df, search_space


def run_sd(mining_df, search_space, depth=4, result_size=20):
    target = ps.BinaryTarget("target", True)

    task = ps.SubgroupDiscoveryTask(
        mining_df,
        target,
        search_space,
        result_set_size=result_size,
        depth=depth,
        qf=ps.StandardQF(0.5)
    )

    result = ps.BeamSearch(beam_width=result_size * 2).execute(task)

    return result

In [ ]:
results = {}

for year, df in dfs.items():
    print(year)
    mining_df, search_space = prepare_mining_df(df, percentile=0.99)
    result = run_sd(mining_df, search_space)
    results[year] = result

results

In [ ]:
results_df = {}

for year, result in results.items():
    results_df[year] = (result.to_dataframe().sort_values("quality", ascending=False))

for year in range(2020, 2025):
    print("\n")
    print("=" * 80)
    print(year)

    display(
        results_df[year][
            ["subgroup", "lift", "coverage_sg", "target_share_sg"]
        ].head(10)
    )

## Experimento 2 - Frequência dos atributos ao longo do tempo

In [ ]:
def attribute_frequency(result_df):
    counter = Counter()

    for subgroup in result_df["subgroup"]:
        subgroup = str(subgroup)
        attrs = re.findall(r"([a-zA-Z_]+)\s*==", subgroup)
        counter.update(attrs)

    return counter


attribute_counts = {}

for year in results_df:
    attribute_counts[year] = attribute_frequency(results_df[year])

rows = []

for year, counter in attribute_counts.items():
    for attr, freq in counter.items():
        rows.append({"year": year, "attribute": attr, "frequency": freq})

attribute_df = pd.DataFrame(rows)
attribute_pivot = (
    attribute_df.pivot(index="attribute", columns="year", values="frequency")
    .fillna(0)
)

plt.figure(figsize=(10, 6))
sns.heatmap(attribute_pivot, annot=True, cmap="YlGnBu")
plt.title("Frequência dos atributos nos Top-20 subgrupos")
plt.show()

## Experimento 3 - Top valores nos subgrupos

In [ ]:
def value_frequency(result_df):
    counter = Counter()

    for subgroup in result_df["subgroup"]:
        subgroup = str(subgroup)
        values = re.findall(r"==\s*'([^']+)'", subgroup)
        counter.update(values)

    return counter


value_counts = {}

for year in results_df:
    value_counts[year] = value_frequency(results_df[year])

for year in sorted(value_counts):
    print("\n" + "=" * 80)
    print(year)

    for value, freq in value_counts[year].most_common(15):
        print(freq, value)

## Experimento 4 - Evolução temporal do melhor subgrupo

In [ ]:
best_subgroups = []

for year, result_df in results_df.items():
    best = result_df.sort_values("lift", ascending=False).iloc[0]

    best_subgroups.append({
        "year": year,
        "subgroup": best["subgroup"],
        "lift": best["lift"],
        "coverage": best["coverage_sg"],
        "target_share": best["target_share_sg"]
    })

best_df = pd.DataFrame(best_subgroups)
best_df

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(best_df["year"], best_df["lift"], marker="o")
plt.title("Lift do melhor subgrupo por ano")
plt.ylabel("Lift")
plt.xlabel("Ano")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(best_df["year"], best_df["coverage"], marker="o")
plt.title("Cobertura do melhor subgrupo por ano")
plt.ylabel("Cobertura")
plt.xlabel("Ano")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(best_df["year"], best_df["target_share"], marker="o")
plt.title("Proporção de positivos no melhor subgrupo")
plt.ylabel("Target Share")
plt.xlabel("Ano")
plt.grid(True)
plt.show()

best_df[["year", "lift", "coverage", "target_share"]].round(3)

## Experimento 5 - Evolução temporal dos setores

In [ ]:
dfs = {
    2020: df_2020,
    2021: df_2021,
    2022: df_2022,
    2023: df_2023,
    2024: df_2024
}

for year, df in dfs.items():
    threshold = df["valor_remuneracao_media"].quantile(0.99)
    df["target_p99"] = (df["valor_remuneracao_media"] >= threshold)

In [ ]:
sector_stats = []

for year, df in dfs.items():
    temp = (
        df.groupby("subsetor_ibge")
        .agg(
            total=("target_p99", "size"),
            positives=("target_p99", "sum"),
            mean_salary=("valor_remuneracao_media", "mean")
        )
        .reset_index()
    )

    temp["share_top1"] = temp["positives"] / temp["total"]
    temp["year"] = year
    sector_stats.append(temp)

sector_stats = pd.concat(sector_stats, ignore_index=True)
sector_stats.head()

In [ ]:
for year in sorted(sector_stats["year"].unique()):
    print("\n" + "=" * 80)
    print(year)

    display(
        sector_stats[sector_stats["year"] == year]
        .sort_values("share_top1", ascending=False)
        [["subsetor_ibge", "share_top1", "mean_salary", "total"]]
        .head(10)
    )

In [ ]:
interesting_sectors = [
    "Instituiçoes de crédito, seguros e capitalizaçao",
    "Administraçao pública direta e autárquica"
]

sector_evolution = sector_stats[sector_stats["subsetor_ibge"].isin(interesting_sectors)]

plt.figure(figsize=(10, 5))

for sector in interesting_sectors:
    subset = sector_evolution[sector_evolution["subsetor_ibge"] == sector]
    plt.plot(subset["year"], subset["share_top1"], marker="o", label=sector)

plt.title("Participação no Top 1% por setor")
plt.xlabel("Ano")
plt.ylabel("% Top 1%")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))

for sector in interesting_sectors:
    subset = sector_evolution[sector_evolution["subsetor_ibge"] == sector]
    plt.plot(subset["year"], subset["mean_salary"], marker="o", label=sector)

plt.title("Salário médio por setor")
plt.xlabel("Ano")
plt.ylabel("Remuneração média")
plt.legend()
plt.grid(True)
plt.show()

## Experimento 6 - Estabilidade dos padrões

In [ ]:
top_subgroups = {}

for year, result_df in results_df.items():
    top_subgroups[year] = set(result_df["subgroup"].head(20))


def jaccard_similarity(set_a, set_b):
    intersection = len(set_a.intersection(set_b))
    union = len(set_a.union(set_b))
    return intersection / union


years = sorted(top_subgroups.keys())

jaccard_matrix = pd.DataFrame(index=years, columns=years, dtype=float)

for y1 in years:
    for y2 in years:
        jaccard_matrix.loc[y1, y2] = jaccard_similarity(top_subgroups[y1], top_subgroups[y2])

jaccard_matrix

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(jaccard_matrix, annot=True, cmap="Blues", vmin=0, vmax=1)
plt.title("Similaridade Jaccard entre Top Subgrupos")
plt.show()

adjacent_years = []

for i in range(len(years) - 1):
    y1 = years[i]
    y2 = years[i + 1]

    adjacent_years.append({
        "transition": f"{y1}-{y2}",
        "jaccard": jaccard_matrix.loc[y1, y2]
    })

adjacent_df = pd.DataFrame(adjacent_years)
adjacent_df

plt.figure(figsize=(8, 4))
sns.barplot(data=adjacent_df, x="transition", y="jaccard")
plt.title("Estabilidade dos Subgrupos entre Anos Consecutivos")
plt.ylabel("Jaccard")
plt.show()

## Experimento 7 - Similaridade Semântica dos Subgrupos

In [ ]:
def subgroup_to_tokens(subgroup):
    subgroup = str(subgroup)
    values = re.findall(r"==\s*'([^']+)'", subgroup)
    return set(values)


semantic_sets = {}

for year, result_df in results_df.items():
    tokens = set()

    for subgroup in result_df["subgroup"].head(20):
        tokens.update(subgroup_to_tokens(subgroup))

    semantic_sets[year] = tokens


semantic_jaccard = pd.DataFrame(index=years, columns=years, dtype=float)

for y1 in years:
    for y2 in years:
        semantic_jaccard.loc[y1, y2] = jaccard_similarity(semantic_sets[y1], semantic_sets[y2])

semantic_jaccard

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(semantic_jaccard, annot=True, cmap="Greens", vmin=0, vmax=1)
plt.title("Similaridade Semântica entre os Top Subgrupos")
plt.show()

comparisons = []

for i in range(len(years) - 1):
    y1 = years[i]
    y2 = years[i + 1]

    comparisons.append({
        "transition": f"{y1}-{y2}",
        "textual_jaccard": jaccard_matrix.loc[y1, y2],
        "semantic_jaccard": semantic_jaccard.loc[y1, y2]
    })

comparison_df = pd.DataFrame(comparisons)
comparison_df

plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_df, x="transition", y="semantic_jaccard")
plt.title("Estabilidade dos Subgrupos entre Anos Consecutivos (Semântico)")
plt.ylabel("Jaccard")
plt.show()

## Experimento 7.5 - Qualidade média dos top-20

In [ ]:
summary = []

for year, df in results_df.items():
    summary.append({
        "year": year,
        "mean_lift": df["lift"].head(20).mean(),
        "mean_target_share": df["target_share_sg"].head(20).mean(),
        "mean_coverage": df["coverage_sg"].head(20).mean()
    })

summary_df = pd.DataFrame(summary)
summary_df

## Experimento 8 - Evolução Regional do Top 1%

In [ ]:
regional_stats = []

for year, df in dfs.items():
    threshold = df["valor_remuneracao_media"].quantile(0.99)

    df = df.copy()
    df["target_p99"] = (df["valor_remuneracao_media"] >= threshold)

    temp = (
        df.groupby("sigla_uf")
        .agg(
            total=("target_p99", "size"),
            positives=("target_p99", "sum"),
            mean_salary=("valor_remuneracao_media", "mean")
        )
        .reset_index()
    )

    temp["share_top1"] = temp["positives"] / temp["total"]
    temp["year"] = year
    regional_stats.append(temp)

regional_stats = pd.concat(regional_stats, ignore_index=True)
regional_stats.head()

In [ ]:
for year in sorted(regional_stats["year"].unique()):
    print("\n" + "=" * 80)
    print(year)

    display(
        regional_stats[regional_stats["year"] == year]
        .sort_values("share_top1", ascending=False)
        [["sigla_uf", "share_top1", "mean_salary", "total"]]
        .head(10)
    )

In [ ]:
regional_pivot = (
    regional_stats.pivot(index="sigla_uf", columns="year", values="share_top1")
    .fillna(0)
)

plt.figure(figsize=(10, 12))
sns.heatmap(regional_pivot, cmap="YlOrRd", annot=True)
plt.title("Participação no Top 1% por Estado")
plt.show()

state_means = (
    regional_stats.groupby("sigla_uf")["share_top1"]
    .mean()
    .sort_values(ascending=False)
)

top_states = state_means.head(5).index.tolist()
top_states

plt.figure(figsize=(10, 5))

for state in top_states:
    subset = regional_stats[regional_stats["sigla_uf"] == state]
    plt.plot(subset["year"], subset["share_top1"], marker="o", label=state)

plt.title("Estados com Maior Participação no Top 1%")
plt.xlabel("Ano")
plt.ylabel("Share Top 1%")
plt.legend()
plt.grid(True)
plt.show()

## Experimento 8.5 - Subgroup Discovery Regional

In [ ]:
def run_state_sd(df, uf):
    state_df = df[df["sigla_uf"] == uf].copy()

    target_col, threshold = create_target(state_df, percentile=0.99)

    search_space = ps.create_selectors(
        state_df,
        ignore=["valor_remuneracao_media", target_col]
    )

    target = ps.BinaryTarget(target_col, True)

    task = ps.SubgroupDiscoveryTask(
        state_df,
        target,
        search_space,
        result_set_size=10,
        depth=2,
        qf=ps.StandardQF(0.5)
    )

    result = ps.BeamSearch(beam_width=20).execute(task)

    return result.to_dataframe().sort_values("quality", ascending=False)

In [ ]:
state_year = {
    "MA": 2022,
    "RO": 2022,
    "MS": 2022,
    "DF": 2024,
    "RJ": 2024,
    "SP": 2024
}

state_results = {}

for state, year in state_year.items():
    print("\n")
    print("=" * 80)
    print(state, year)

    result_df = run_state_sd(dfs[year], state)
    state_results[state] = result_df

    display(
        result_df[["subgroup", "lift", "coverage_sg", "target_share_sg"]]
    )

In [ ]:
rows = []

for state, result_df in state_results.items():
    counter = attribute_frequency(result_df)

    for attr, freq in counter.items():
        rows.append({"state": state, "attribute": attr, "frequency": freq})

attribute_df = pd.DataFrame(rows)
attribute_pivot = (
    attribute_df.pivot(index="attribute", columns="state", values="frequency")
    .fillna(0)
)

attribute_pivot

plt.figure(figsize=(12, 6))
sns.heatmap(attribute_pivot, annot=True, cmap="YlGnBu")
plt.title("Frequência dos atributos nos Top Subgrupos por Estado")
plt.show()

In [ ]:
for state in state_results:
    print("\n")
    print("=" * 80)
    print(state)

    counter = value_frequency(state_results[state])

    for value, freq in counter.most_common(15):
        print(freq, value)

## Experimento 9 - Evolução da importância dos atributos

In [ ]:
def attribute_importance(df):
    df = df.copy()

    target_col, _ = create_target(df, 0.99)

    X = df.drop(columns=["valor_remuneracao_media", target_col])
    y = df[target_col]

    X_encoded = pd.DataFrame()

    for col in X.columns:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X[col].astype(str))

    mi = mutual_info_classif(
        X_encoded,
        y,
        discrete_features=True,
        random_state=42
    )

    return pd.DataFrame({
        "attribute": X.columns,
        "mutual_information": mi
    }).sort_values("mutual_information", ascending=False)


importance_results = []

for year, df in dfs.items():
    imp = attribute_importance(df)
    imp["year"] = year
    importance_results.append(imp)

importance_df = pd.concat(importance_results, ignore_index=True)
importance_df.head()

In [ ]:
importance_df

importance_pivot = importance_df.pivot(
    index="attribute",
    columns="year",
    values="mutual_information"
)

plt.figure(figsize=(10, 6))
sns.heatmap(importance_pivot, annot=True, cmap="YlGnBu")
plt.title("Importância dos atributos para identificar o Top 1%")
plt.show()